In [ ]:
#!/usr/bin/env python3
"""
Patch script — add oldret and new IRASA columns to existing session CSVs
------------------------------------------------------------------------
New windows:
  oldret : -750  to    0 ms  (buffer -1250 to  +500 ms)
  new    : -1000 to +500 ms  (buffer -1500 to +1000 ms)

New columns (18 each, 72 total):
  oldret_frac_f00..f17,  oldret_osc_f00..f17
  new_frac_f00..f17,     new_osc_f00..f17

Target dir : /home1/rcao92/Spatial_Binding/subject_results_HP_LTC_DBOY1
Skip logic : if 'oldret_frac_f00' AND 'new_frac_f00' already present → skip
Parallelism: session-level (same N_WORKERS logic as main pipeline)

Row-matching strategy (mismatch-safe):
  Clean recalls are re-derived per trial using the exact same coordinate-
  matching + make_clean_recalls_mask2d logic as the main pipeline.
  Values are then written back to the CSV matched on the composite key:
      (trial, recall_output_position, region, pair_idx)
  This is fully order-independent and robust to intrusions/repeats.
"""

import os
import copy
import traceback
import warnings
from pathlib import Path
from multiprocessing import Pool, cpu_count

import numpy as np
import pandas as pd
from scipy.spatial.distance import euclidean

warnings.filterwarnings('ignore')

# ============================================================================
# IMPORTS
# ============================================================================
try:
    from irasa.IRASA import IRASA, SSL_transform
    print("✓ IRASA imported successfully")
except ImportError as e:
    print(f"✗ IRASA not available: {e}")
    def SSL_transform(x):
        return np.sign(x) * np.log10(1 + np.abs(x))

import cmlreaders as cml
print("✓ cmlreaders imported successfully")

# ============================================================================
# CONFIGURATION
# ============================================================================

EXPERIMENT = 'DBOY1'
OUTPUT_DIR = Path('/home1/rcao92/Spatial_Binding/subject_results_HP_LTC_DBOY1')

IRASA_HSET  = np.arange(1.1, 1.951, 0.05)
IRASA_FREQS = np.round(np.logspace(np.log10(3), np.log10(128), 18), 2)
N_FREQS     = len(IRASA_FREQS)

REGIONS = ['LHP', 'RHP', 'LLTC', 'RLTC']

# oldret: -750 to 0 ms, 500 ms buffer each side
OLDRET_WIN_START_MS = -750
OLDRET_WIN_END_MS   =    0
OLDRET_EPOCH_START  = -1250   # -750 - 500
OLDRET_EPOCH_END    =   500   #    0 + 500

# new: -1000 to +500 ms, 500 ms buffer each side
NEW_WIN_START_MS = -1000
NEW_WIN_END_MS   =   0
NEW_EPOCH_START  = -1500   # -1000 - 500
NEW_EPOCH_END    =  1000   #  +500 + 500

_slurm_cpus = os.environ.get('SLURM_CPUS_PER_TASK')
N_WORKERS = int(_slurm_cpus) if _slurm_cpus else min(cpu_count(), 12)

# ============================================================================
# COLUMN NAMES
# ============================================================================

OLDRET_FRAC_COLS = [f'oldret_frac_f{i:02d}' for i in range(N_FREQS)]
OLDRET_OSC_COLS  = [f'oldret_osc_f{i:02d}'  for i in range(N_FREQS)]
NEW_FRAC_COLS    = [f'new_frac_f{i:02d}'    for i in range(N_FREQS)]
NEW_OSC_COLS     = [f'new_osc_f{i:02d}'     for i in range(N_FREQS)]
PATCH_COLS       = OLDRET_FRAC_COLS + OLDRET_OSC_COLS + NEW_FRAC_COLS + NEW_OSC_COLS

# ============================================================================
# HELPERS  (verbatim from main pipeline)
# ============================================================================

def ssl(x):
    return SSL_transform(x)

def ms_to_sample(ms, epoch_start_ms, sfreq):
    return int(round((ms - epoch_start_ms) / 1000.0 * sfreq))

def convert_recalled_coords_to_itemnos(presented_coords, recalled_coords, tolerance=0.01):
    itemnos = []
    for rc in recalled_coords:
        dists = np.array([euclidean(rc, pc) for pc in presented_coords])
        ci    = np.argmin(dists)
        itemnos.append(ci + 1 if dists[ci] <= tolerance else -1)
    return np.array(itemnos)

def make_recalls_matrix(pres_itemnos, rec_itemnos):
    n_trials  = pres_itemnos.shape[0]
    n_recalls = rec_itemnos.shape[1]
    recalls   = np.zeros([n_trials, n_recalls], dtype=int)
    for t in range(n_trials):
        for r in range(n_recalls):
            v = rec_itemnos[t, r]
            if v == 0 or np.isnan(v):
                continue
            elif v > 0:
                sp = np.where(v == pres_itemnos[t, :])[0] + 1
                recalls[t, r] = sp[0] if len(sp) == 1 else -1
            else:
                recalls[t, r] = -1
    return recalls

def make_clean_recalls_mask2d(data):
    result = copy.deepcopy(data)
    for num, item in enumerate(data):
        seen = []
        for index, recall in enumerate(item):
            if recall > 0 and recall not in seen:
                result[num][index] = 1
                seen.append(recall)
            else:
                result[num][index] = 0
    return result

def define_region_masks(pairs_df):
    n = len(pairs_df)
    masks = {}

    region_col = None
    for col_candidate in ['mni.region', 'ind.region', 'stein.region',
                          'avg.region', 'dk.region']:
        if col_candidate in pairs_df.columns:
            region_col = col_candidate
            break
    if region_col is None:
        region_cols = [c for c in pairs_df.columns if 'region' in c.lower()]
        region_col  = region_cols[0] if region_cols else None

    if region_col is None:
        for r in REGIONS:
            masks[r] = np.zeros(n, dtype=bool)
        return masks

    regions_str = pairs_df[region_col].fillna('')

    def is_left(row_idx):
        if 'label' in pairs_df.columns:
            lbl = str(pairs_df['label'].iloc[row_idx])
            if lbl and lbl[0].upper() == 'L':
                return True
            if lbl and lbl[0].upper() == 'R':
                return False
        return 'left' in regions_str.iloc[row_idx].lower()

    left_mask  = np.array([is_left(i) for i in range(n)])
    right_mask = ~left_mask

    hipp_mask = regions_str.str.contains('Hippocampus', case=False, na=False).values
    masks['LHP'] = hipp_mask & left_mask
    masks['RHP'] = hipp_mask & right_mask

    ltc_mask = regions_str.str.contains(
        r'Superior Temporal|Middle Temporal|Inferior Temporal|STG|MTG|ITG',
        case=False, na=False, regex=True
    ).values
    masks['LLTC'] = ltc_mask & left_mask
    masks['RLTC'] = ltc_mask & right_mask

    return masks

def compute_irasa_channel(sig, sfreq):
    ir   = IRASA(sig=sig.astype(float), freqs=IRASA_FREQS, samplerate=sfreq,
                 hset=IRASA_HSET, flag_filter=1, flag_detrend=1)
    frac = ir.fractal
    osc  = ir.mixed - frac
    return ssl(frac), ssl(osc)

def irasa_window(eeg_win, sfreq, unique_ch_indices):
    """Compute IRASA for every channel across all recall events in eeg_win."""
    n     = eeg_win.shape[0]
    cache = {}
    for ch_idx in unique_ch_indices:
        frac_rows, osc_rows = [], []
        for ri in range(n):
            sig = eeg_win[ri, ch_idx, :]
            try:
                f, o = compute_irasa_channel(sig, sfreq)
            except Exception:
                f = np.full(N_FREQS, np.nan)
                o = np.full(N_FREQS, np.nan)
            frac_rows.append(f)
            osc_rows.append(o)
        cache[ch_idx] = {'frac': np.array(frac_rows), 'osc': np.array(osc_rows)}
    return cache

# ============================================================================
# PER-SESSION WORKER
# ============================================================================

def patch_session(csv_path):
    """
    Load one session CSV, compute oldret + new IRASA windows, append
    columns, overwrite in place.

    Row-matching is mismatch-safe: clean recalls are re-derived via the
    same coordinate-matching logic as the main pipeline, then values are
    written back using the composite key (trial, recall_output_position,
    region, pair_idx) — fully order-independent.
    """
    msgs     = []
    csv_path = Path(csv_path)
    tag      = csv_path.stem

    # ── Skip if already fully patched ─────────────────────────────────────
    df = pd.read_csv(csv_path)
    if 'oldret_frac_f00' in df.columns and 'new_frac_f00' in df.columns:
        msgs.append(f"   [{tag}] Already patched – skipping")
        return '\n'.join(msgs), 0

    # ── Parse subject / session from filename ─────────────────────────────
    # Pattern: {subject}_{session}_DBOY1_irasa_HP_LTC_wide.csv
    parts   = csv_path.stem.split('_')
    subject = parts[0]
    session = int(parts[1])

    msgs.append(f"   [{tag}] Loaded {len(df):,} rows")

    try:
        reader       = cml.CMLReader(subject, EXPERIMENT, session=session)
        pairs_df     = reader.load('pairs')
        region_masks = define_region_masks(pairs_df)

        channel_info = []
        for region in REGIONS:
            for ch_idx in np.where(region_masks[region])[0]:
                row_p = pairs_df.iloc[int(ch_idx)]
                label = str(row_p.get('label', row_p.get('tagName', f'ch{ch_idx}')))
                channel_info.append((region, int(ch_idx), label))

        if len(channel_info) == 0:
            msgs.append(f"   [{tag}] No relevant channels found – skipping")
            return '\n'.join(msgs), 0

        unique_ch_indices = list({ci for _, ci, _ in channel_info})

        evs = reader.load('task_events')

        # Pre-allocate new columns as NaN (only those not already present)
        for col in PATCH_COLS:
            if col not in df.columns:
                df[col] = np.nan

        # ── Per-trial processing ──────────────────────────────────────────
        for trial in sorted(df['trial'].unique()):

            trial_evs         = evs[evs['trial'] == trial]
            trial_evs_RECWORD = trial_evs[trial_evs['type'] == 'REC_WORD']
            trial_evs_WORD    = trial_evs[trial_evs['type'] == 'WORD']

            if len(trial_evs_RECWORD) == 0 or len(trial_evs_WORD) == 0:
                continue

            try:
                # ── Re-derive clean recalls (identical to main pipeline) ───
                # Re-running coordinate matching guarantees that local_idx
                # 0,1,2,… maps to recall_output_position 1,2,3,… in the CSV.
                recalled_coords  = np.column_stack([trial_evs_RECWORD['storeX'],
                                                    trial_evs_RECWORD['storeZ']])
                presented_coords = np.column_stack([trial_evs_WORD['storeX'],
                                                    trial_evs_WORD['storeZ']])
                rec_itemnos_array = convert_recalled_coords_to_itemnos(
                    presented_coords, recalled_coords
                )

                pres_2d = np.array([list(range(1, len(presented_coords) + 1))])
                rec_2d  = np.zeros((1, len(presented_coords)), dtype=int)
                rec_2d[0, :len(rec_itemnos_array)] = rec_itemnos_array

                clean_mask        = make_clean_recalls_mask2d(
                    make_recalls_matrix(pres_2d, rec_2d))[0]
                clean_recall_idx  = np.where(clean_mask == 1)[0]
                clean_rec_itemnos = rec_itemnos_array[clean_recall_idx]

                if len(clean_rec_itemnos) == 0:
                    continue

                # clean_RECWORD_evs is in the same order the main pipeline used
                clean_RECWORD_evs = trial_evs_RECWORD.iloc[
                    clean_recall_idx].reset_index(drop=True)

                # ── Load EEG for oldret window ────────────────────────────
                eeg_oldret_cont = reader.load_eeg(
                    clean_RECWORD_evs,
                    OLDRET_EPOCH_START, OLDRET_EPOCH_END,
                    scheme=pairs_df
                )
                sfreq          = float(eeg_oldret_cont.samplerate)
                s0_or = ms_to_sample(OLDRET_WIN_START_MS, OLDRET_EPOCH_START, sfreq)
                s1_or = ms_to_sample(OLDRET_WIN_END_MS,   OLDRET_EPOCH_START, sfreq)
                eeg_oldret_win = eeg_oldret_cont.data[:, :, s0_or:s1_or]

                # ── Load EEG for new window ───────────────────────────────
                eeg_new_cont = reader.load_eeg(
                    clean_RECWORD_evs,
                    NEW_EPOCH_START, NEW_EPOCH_END,
                    scheme=pairs_df
                )
                s0_nw = ms_to_sample(NEW_WIN_START_MS, NEW_EPOCH_START, sfreq)
                s1_nw = ms_to_sample(NEW_WIN_END_MS,   NEW_EPOCH_START, sfreq)
                eeg_new_win = eeg_new_cont.data[:, :, s0_nw:s1_nw]

                # ── IRASA for both windows ────────────────────────────────
                oldret_irasa = irasa_window(eeg_oldret_win, sfreq, unique_ch_indices)
                new_irasa    = irasa_window(eeg_new_win,    sfreq, unique_ch_indices)

                # ── Write back using composite key — order-independent ─────
                for local_idx in range(len(clean_rec_itemnos)):
                    out_pos = local_idx + 1   # recall_output_position (1-based)

                    base_mask = (
                        (df['trial'] == trial) &
                        (df['recall_output_position'] == out_pos)
                    )

                    for region, ch_idx, _ in channel_info:
                        cell_mask = (
                            base_mask &
                            (df['region']   == region) &
                            (df['pair_idx'] == ch_idx)
                        )
                        if not cell_mask.any():
                            continue

                        # Guard: IRASA array must contain this local_idx
                        if local_idx >= oldret_irasa[ch_idx]['frac'].shape[0]:
                            continue

                        frac_or = oldret_irasa[ch_idx]['frac'][local_idx]
                        osc_or  = oldret_irasa[ch_idx]['osc'][local_idx]
                        frac_nw = new_irasa[ch_idx]['frac'][local_idx]
                        osc_nw  = new_irasa[ch_idx]['osc'][local_idx]

                        for fi in range(N_FREQS):
                            df.loc[cell_mask, f'oldret_frac_f{fi:02d}'] = frac_or[fi]
                            df.loc[cell_mask, f'oldret_osc_f{fi:02d}']  = osc_or[fi]
                            df.loc[cell_mask, f'new_frac_f{fi:02d}']    = frac_nw[fi]
                            df.loc[cell_mask, f'new_osc_f{fi:02d}']     = osc_nw[fi]

            except Exception as e:
                msgs.append(f"   [{tag}] Trial {trial} FAILED: {e}")
                msgs.append(traceback.format_exc())
                continue

        # ── Overwrite CSV ─────────────────────────────────────────────────
        df.to_csv(csv_path, index=False)
        msgs.append(
            f"   ✓ [{tag}] Saved ({len(df):,} rows, "
            f"{len(PATCH_COLS)} new columns added)"
        )

    except Exception as e:
        msgs.append(f"   [{tag}] FAILED: {e}")
        msgs.append(traceback.format_exc())
        return '\n'.join(msgs), 0

    return '\n'.join(msgs), len(df)

# ============================================================================
# MAIN
# ============================================================================

if __name__ == '__main__':

    print(f"\n{'='*80}")
    print(f"PATCH: adding oldret (-750→0 ms) + new (-1000→+500 ms) — {EXPERIMENT}")
    print(f"{'='*80}")

    all_csvs     = sorted(OUTPUT_DIR.glob(f"*_{EXPERIMENT}_irasa_HP_LTC_wide.csv"))
    session_csvs = [f for f in all_csvs if not f.name.startswith('ALL_SUBJECTS')]

    pending      = []
    already_done = 0
    for f in session_csvs:
        df_head = pd.read_csv(f, nrows=1)
        if 'oldret_frac_f00' in df_head.columns and 'new_frac_f00' in df_head.columns:
            already_done += 1
        else:
            pending.append(f)

    print(f"Session CSVs found  : {len(session_csvs)}")
    print(f"Already patched     : {already_done}")
    print(f"To patch            : {len(pending)}")
    print(f"Workers             : {N_WORKERS}")
    print(f"New columns         : {PATCH_COLS[0]} … {PATCH_COLS[-1]}"
          f"  ({len(PATCH_COLS)} total)")
    print("=" * 80)

    if pending:
        with Pool(processes=N_WORKERS) as pool:
            results = pool.map(patch_session, pending)

        print(f"\n{'='*80}")
        print("PATCH RESULTS")
        print(f"{'='*80}")
        total_rows = 0
        for msg, n_rows in results:
            print(msg)
            total_rows += n_rows
        print(f"\nTotal rows patched this run: {total_rows:,}")
    else:
        print("All session CSVs already patched – nothing to do.")

    # ── Rebuild master CSV ────────────────────────────────────────────────
    print(f"\n{'='*80}")
    print(f"REBUILDING MASTER CSV — {EXPERIMENT}")
    print(f"{'='*80}")

    all_session_csvs = sorted(OUTPUT_DIR.glob(f"*_{EXPERIMENT}_irasa_HP_LTC_wide.csv"))
    all_session_csvs = [f for f in all_session_csvs
                        if not f.name.startswith('ALL_SUBJECTS')]

    if all_session_csvs:
        master_df   = pd.concat(
            [pd.read_csv(f) for f in all_session_csvs],
            ignore_index=True
        )
        master_path = OUTPUT_DIR / f"ALL_SUBJECTS_{EXPERIMENT}_irasa_HP_LTC_wide.csv"
        master_df.to_csv(master_path, index=False)
        print(f"  Subjects : {master_df['subject'].nunique()}")
        print(f"  Sessions : {master_df['session'].nunique()}")
        print(f"  Channels : {master_df['channel_label'].nunique()}")
        print(f"  Rows     : {len(master_df):,}")
        print(f"  Columns  : {len(master_df.columns)}")
        print(f"  → {master_path}")
    else:
        print("  No session CSVs found – master not written")

    print(f"\n{'='*80}")
    print("✓ PATCH COMPLETE")
    print(f"{'='*80}")

✓ IRASA imported successfully
✓ cmlreaders imported successfully

PATCH: adding oldret (-750→0 ms) + new (-1000→+500 ms) — DBOY1
Session CSVs found  : 85
Already patched     : 0
To patch            : 85
Workers             : 12
New columns         : oldret_frac_f00 … new_osc_f17  (72 total)
Removing linear trend
Filtering to avoid aliasing
Computing fractal PSD
Removing linear trend
Removing linear trend
Filtering to avoid aliasing
Computing fractal PSD
Filtering to avoid aliasing
Computing fractal PSD
Time elapsed for FFT: 0.6266 s
Removing linear trend
Filtering to avoid aliasing
Computing fractal PSD
Time elapsed for FFT: 0.5290 s
Removing linear trend
Time elapsed for FFT: 0.6689 s
Removing linear trend
Filtering to avoid aliasing
Filtering to avoid aliasing
Computing fractal PSD
Computing fractal PSD
Time elapsed for FFT: 0.6257 s
Removing linear trend
Filtering to avoid aliasing
Computing fractal PSD
Time elapsed for FFT: 0.4075 s
Removing linear trend
Filtering to avoid aliasing

In [5]:
pwd

'/home1/rcao92/Spatial_Binding/subject_results_HP_LTC_DBOY1'